# Lecture 2 Linear Regression Code Practice

### Importing Numpy and Creating a Dummy data set with 100 samples, and 1 feature

In [ ]:
import numpy as np

# 1. Make m training examples of x (just random numbers rn)
m = 100                       # m = number of training examples (the Sigma over examples runs 1..m)
x = np.random.randn(m, 1)     # (100, 1) - one feature value per example; n = 1 feature

# 2. Make y from the KNOWN rule: y = 5 + 2x + noise (true theta0 = 5 intercept, true theta1 = 2 slope)
noise = np.random.randn(m, 1) # (100, 1) - small random wiggle, same shape as x
y = 5 + 2 * x + noise         # (100, 1) - one target per example; goal is to recover theta approx [5, 2]

# 3. Building X = the design matrix: a column of 1s next to x
# (the 1s column is the intercept term x0 = 1, so theta0 is learned inside the same matmul)
X = np.c_[np.ones((m,1)), x]  # (100, 2) - rows = examples, cols = [x0=1, x1]; #params = #cols of X

# 4. initializing theta (2 parameters: theta0 intercept + theta1 slope)
theta = np.random.randn(2, 1) # (2, 1) - the PARAMETER vector theta (n+1 = 2 rows). NOT the hypothesis itself

# 5. Predictions = the hypothesis h_theta(x) = Sigma_j theta_j x_j
predictions = X @ theta       # (100, 2) @ (2, 1) -> (100, 1); the @ does the multiply-and-sum over features j

## Cost Function

In [ ]:
def compute_cost(X, y, theta):
  m = len(y)                              # number of examples (the Sigma_i runs over these)
  predictions = X @ theta                 # (100, 1) - h_theta(x) for every example at once
  errors = (predictions - y)              # (100, 1) - residual (h(x) - y) per example
  cost = (1/(2*m)) * (errors ** 2).sum()  # J(theta) = (1/2m) Sigma_i (h(x^i) - y^i)^2
                                          # errors**2 is element-wise -> .sum() is the Sigma over examples i.
                                          # @ can't do this sum: squaring is element-wise, not a dot product
  return cost                             # scalar - one number = how wrong theta is, averaged over examples

compute_cost(X, y, theta)

## 1. Gradient Descent : **The main event**

When someone says "gradient descent" with no qualifier, batch is the default. The plain-vanilla version that uses the whole dataset each step.

Batch Gradient Descent (The Smooth Path)

In [ ]:
def gradient_descent(X, y, theta, alpha, num_iters):
  m = len(y)                       # number of examples averaged over
  cost_history=[]                  # track cost each step to watch it drop

  for i in range(num_iters):       # i indexes ITERATIONS (full-dataset passes), NOT examples
    predictions = X @ theta        # (100, 1) - h_theta(x) for all m examples
    errors = predictions - y       # (100, 1) - residual (h(x) - y), same as in the cost
    gradient = (1/m) * X.T @ errors  # dJ/dtheta_j = (1/m) Sigma_i (h(x^i)-y^i) x_j^i  -> shape (2, 1)
                                     # X.T is (2, 100): features in rows now, so (2,100)@(100,1) multiplies
                                     # each residual by x_j and sums over all examples. The @ IS the Sigma_i,
                                     # which is why there is no separate .sum() here (dot = multiply-and-sum)
    theta = theta - alpha * gradient # theta := theta - alpha * dJ/dtheta; (2,1) - (2,1) -> (2,1).
                                     # MINUS: the gradient points uphill, so we step against it to go downhill

    cost_history.append(compute_cost(X, y, theta)) # log the cost; should decrease monotonically

  return theta, cost_history

theta_final, history = gradient_descent(X, y, theta, 0.1, 1000)
theta_final                          # approx [[5.05],[2.08]] - recovers true [5, 2]; matches the normal eqn exactly

## 2. Stochastic Gradient Descent (The Drunken Walk)

Instead of looking at all 100 rows of X at once, Stochastic GD picks one random row at a time, calculates the gradient for just that single point, and immediately updates the theta

In [ ]:
def stochastic_gradient_descent(X, y, theta, alpha, num_epochs):
  m = len(y)
  cost_history = []

  for epoch in range(num_epochs):       # one epoch = one full sweep through all m examples
    for i in range(m):                  # i indexes EXAMPLES here - we update once per single example
      xi = X[i:i+1] # (1, 2) - row i kept 2D via slice (X[i] alone would collapse to (2,))
      yi = y[i:i+1] # (1, 1) - target i kept 2D (single value, not (1, 2))

      predictions = xi @ theta          # (1, 2) @ (2, 1) -> (1, 1); hypothesis for just this one example
      error = predictions - yi          # (1, 1) - the single residual (h(x^i) - y^i)
      gradient = xi.T @ error           # (2, 1) @ (1, 1) -> (2, 1); (h(x^i)-y^i) x_j^i for THIS example only.
                                        # NO 1/m factor: it's one example, not an average over m
      theta = theta - alpha * gradient  # (2, 1) - same update rule theta := theta - alpha*grad, per example

    cost_history.append(compute_cost(X, y, theta)) # log full-dataset cost once per epoch
  return theta, cost_history

theta_final, history = stochastic_gradient_descent(X, y, theta, 0.1, 1000)
theta_final                             # approx [[5.21],[2.20]] - close to [5, 2] but slightly off (sampling jitter)

## 3. The Ultimate Shortcut (The Normal Equation)

To get the global minimum directly without any loops

In [ ]:
# Normal equation: theta = (X^T X)^-1 X^T y - solves dJ/dtheta = 0 exactly in one shot (no loops, no alpha)
theta_normal = np.linalg.inv(X.T @ X) @ X.T @ y  # (2,2)^-1 @ (2,100) @ (100,1) -> (2, 1)
                                                 # X.T@X is (2,2); inverting it is O(n^3) - fine for n=1,
                                                 # infeasible for huge n, and fails if X.T@X is singular
theta_normal                                     # approx [[5.05],[2.08]] - IDENTICAL to batch GD (both hit the global optimum)